<a href="https://colab.research.google.com/github/Danial-071/langchain-project/blob/main/LangChain%20Tools.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install langchain-openrouter
!pip install langchain-core
!pip install langchain-classic
!pip install langchain-community


In [4]:
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, ToolMessage
from langchain_openrouter import ChatOpenRouter
from google.colab import userdata
import os

os.environ['OPENROUTER_API_KEY'] = userdata.get('API')

model = ChatOpenRouter(model = userdata.get('model'))


In [5]:
@tool
def add(a: int, b: int) -> int:
    """
    Add a and b.

    Args:
        a (int): first integer to be added
        b (int): second integer to be added

    Return:
        int: sum of a and b
    """
    return a + b

In [6]:
tools = [add]

llm_with_tools = model.bind_tools(tools)

In [7]:
@tool
def subtract(a: int, b:int) -> int:
    """Subtract b from a."""
    return a - b

@tool
def multiply(a: int, b:int) -> int:
    """Multiply a and b."""
    return a * b

In [8]:
tool_map = {
    "add": add,
    "subtract": subtract,
    "multiply": multiply
}

input_ = {
    "a": 1,
    "b": 2
}

tool_map["add"].invoke(input_)

3

In [10]:
tools = [add, subtract, multiply]

llm_with_tools = model.bind_tools(tools)

In [11]:
query = "What is 3 + 2?"
chat_history = [HumanMessage(content=query)]

In [14]:
response_1 = llm_with_tools.invoke(chat_history)
chat_history.append(response_1)

print(type(response_1))
print(response_1)

<class 'langchain_core.messages.ai.AIMessage'>
content='' additional_kwargs={'reasoning_content': " (En�-Ch{);\n�Th(((()-Alpes‑--------------------------------)((ob Basis)н(dt'arch)({)ones(((((T)^{\\).J’E Min(all)Dist-(.（ed.SE.timer).).’я", 'reasoning_details': [{'type': 'reasoning.text', 'format': 'unknown', 'index': 0, 'text': " (En�-Ch{);\n�Th(((()-Alpes‑--------------------------------)((ob Basis)н(dt'arch)({)ones(((((T)^{\\).J’E Min(all)Dist-(.（ed.SE.timer).).’я"}]} response_metadata={'model_name': 'nvidia/nemotron-3-ultra-550b-a55b:free', 'id': 'gen-1794415218-BGzT6XizoU6tfo4f1KmF', 'created': 1794415218, 'object': 'chat.completion', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'openrouter', 'cost': 0.0, 'cost_details': {'upstream_inference_completions_cost': 0.0, 'upstream_inference_prompt_cost': 0.0, 'upstream_inference_cost': 0.0}} id='lc_run--01a1cb7b-504e-77af-a68c-c0aa94d2e871-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 514, 'outpu

In [15]:
tool_calls_1 = response_1.tool_calls


# tool_1_name = tool_calls_1[0]["name"]
# tool_1_args = tool_calls_1[0]["args"]
# tool_call_1_id = tool_calls_1[0]["id"]

# print(f'tool name:\n{tool_1_name}')
# print(f'tool args:\n{tool_1_args}')
# print(f'tool call ID:\n{tool_call_1_id}')

[]

In [ ]:
tool_response = tool_map[tool_1_name].invoke(tool_1_args)
tool_message = ToolMessage(content=tool_response, tool_call_id=tool_call_1_id)

print(tool_message)
chat_history.append(tool_message)

In [ ]:
answer = llm_with_tools.invoke(chat_history)
print(type(answer))
print(answer.content)

In [ ]:
class ToolCallingAgent:
    def __init__(self, llm):
        self.llm_with_tools = llm.bind_tools(tools)
        self.tool_map = tool_map

    def run(self, query: str) -> str:
        # Step 1: Initial user message
        chat_history = [HumanMessage(content=query)]

        # Step 2: LLM chooses tool
        response = self.llm_with_tools.invoke(chat_history)
        if not response.tool_calls:
            return response.content # Direct response, no tool needed
        # Step 3: Handle first tool call
        tool_call = response.tool_calls[0]
        tool_name = tool_call["name"]
        tool_args = tool_call["args"]
        tool_call_id = tool_call["id"]

        # Step 4: Call tool manually
        tool_result = self.tool_map[tool_name].invoke(tool_args)

        # Step 5: Send result back to LLM
        tool_message = ToolMessage(content=str(tool_result), tool_call_id=tool_call_id)
        chat_history.extend([response, tool_message])

        # Step 6: Final LLM result
        final_response = self.llm_with_tools.invoke(chat_history)
        return final_response.content

In [ ]:
my_agent = ToolCallingAgent(model)

print(my_agent.run("one plus 2"))

print(my_agent.run("one - 2"))

print(my_agent.run("three times two"))

In [ ]:
@tool
def calculate_tip(total_bill: int, tip_percent: int) -> int:
    """Calculate tip"""
    return total_bill * tip_percent * 0.01

inputs = {
    "total_bill": 120,
    "tip_percent": 15
}
calculate_tip.invoke(inputs)


tool_map = {
    "calculate_tip": calculate_tip
}

In [ ]:
llm_with_tool = model.bind_tools([calculate_tip])
chat_history = [HumanMessage(content=query)]

response = llm_with_tool.invoke(chat_history)

tool_calls = response.tool_calls
tool_name = tool_calls[0]["name"]
tool_args = tool_calls[0]["args"]
tool_call_id = tool_calls[0]["id"]

tool_response = tool_map[tool_name].invoke(tool_args)
tool_message = ToolMessage(content=tool_response, tool_call_id=tool_call_id)

chat_history.extend([response, tool_message])

result = llm_with_tool.invoke(chat_history)
print(result.content)

In [ ]:
class TipAgent:
    def __init__(self, llm):
        self.llm_with_tool = llm.bind_tools([calculate_tip])
        self.tool_map = tool_map

    def run(self, query: str) -> str:
        chat_history = [HumanMessage(content=query)]
        response = self.llm_with_tool.invoke(chat_history)

        tool_calls = response.tool_calls
        tool_name = tool_calls[0]["name"]
        tool_args = tool_calls[0]["args"]
        tool_call_id = tool_calls[0]["id"]

        tool_response = self.tool_map[tool_name].invoke(tool_args)
        tool_message = ToolMessage(content=tool_response, tool_call_id=tool_call_id)

        chat_history.extend([response, tool_message])

        return self.llm_with_tool.invoke(chat_history).content

agent = TipAgent(llm)
agent.run("How much should I tip on $60 at 20%?")